# AAI614: Data Science & its Applications

*Notebook 3.2: Practice with Data Cleaning*

<a href="https://colab.research.google.com/github/techseeko/AAI614_Haidar/blob/main/Week-3/Saoud-Notebook3.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

Exercise I. Load the following datafile from GitHub

In [14]:
grads = pd.read_csv("https://raw.githubusercontent.com/harmanani/AAI614/main/Week%203/grads.csv")

In [15]:
grads

,Student Name,Avg Hours Studies per Week,GPA,University,Sense of Humour (0-5),Salary
0,George,20,NaN,NYU,3.0,$40k
1,Jerry,35,3.5,Columbia,5.0,$80k
2,Elaine,55,4.0,Columbia,4.2,$60k
3,Cosmo,5,2.0,City College,2.0,$25k
4,Newman,25,2.8,City College,0.0,$50k
5,Frank,35,3.0,Festivus Uni,NaN,$40k
6,Estelle,100,3.2,Festivus Uni,1.7,$0k
7,Leo,15,2.4,Festivus Uni,0.0,$35k
8,Rachel,50,4.0,Columbia,NaN,$75k


Question 1: Identify all the outliers in the above data.  Justify your answers using objective measures.

In [16]:
# Make a copy of the DataFrame to avoid modifying the original 'grads' in place during initial processing
grads_outliers = grads.copy()

# Convert 'Salary' column to numeric, removing '$' and 'k' and handling missing values
grads_outliers['Salary'] = grads_outliers['Salary'].replace({r'\$': '', 'k': ''}, regex=True).astype(float) * 1000

# Define a function to detect outliers using the IQR method
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Numerical columns to check for outliers
numerical_cols = ['Avg Hours Studies per Week', 'GPA', 'Sense of Humour (0-5)', 'Salary']

print("\n--- Outlier Detection Results (IQR Method) ---")
all_outliers = pd.DataFrame()

for col in numerical_cols:
    # Drop NaN values for outlier detection in the specific column, to avoid errors in quantile calculation
    # and to ensure outliers are detected only among existing data points.
    temp_df = grads_outliers.dropna(subset=[col])
    if not temp_df.empty:
        outliers, lower_bound, upper_bound = detect_outliers_iqr(temp_df, col)
        if not outliers.empty:
            print(f"\nColumn: '{col}'")
            print(f"  IQR Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
            print("  Outliers found:")
            print(outliers[['Student Name', col]])
            all_outliers = pd.concat([all_outliers, outliers])
        else:
            print(f"\nNo outliers found in '{col}' using the IQR method.")
    else:
        print(f"\nSkipping '{col}' due to all values being NaN.")

# Display unique outlier rows
if not all_outliers.empty:
    print("\n--- All unique outlier rows identified ---")
    print(all_outliers.drop_duplicates())
else:
    print("\nNo outliers were identified across any of the numerical columns.")

print("\nJustification: The Interquartile Range (IQR) method defines outliers as data points that fall below Q1 - 1.5 * IQR or above Q3 + 1.5 * IQR. This method is robust to skewed distributions and is a commonly accepted statistical approach for outlier detection. The displayed rows are the data points that satisfy this criterion for the respective columns.")



--- Outlier Detection Results (IQR Method) ---

Column: 'Avg Hours Studies per Week'
  IQR Bounds: [-25.00, 95.00]
  Outliers found:
  Student Name  Avg Hours Studies per Week
6      Estelle                         100

No outliers found in 'GPA' using the IQR method.

No outliers found in 'Sense of Humour (0-5)' using the IQR method.

No outliers found in 'Salary' using the IQR method.

--- All unique outlier rows identified ---
  Student Name  Avg Hours Studies per Week  GPA     University  \
6      Estelle                         100  3.2  Festivus Uni    

   Sense of Humour (0-5)  Salary  
6                    1.7     0.0  

Justification: The Interquartile Range (IQR) method defines outliers as data points that fall below Q1 - 1.5 * IQR or above Q3 + 1.5 * IQR. This method is robust to skewed distributions and is a commonly accepted statistical approach for outlier detection. The displayed rows are the data points that satisfy this criterion for the respective columns.


Question 2: There are various data that are missing.  Fill-in the missing data or delete the rows/columns that you think you should delete.  Justify your answer

In [17]:
# Create a copy of the original DataFrame to work with, preserving the original 'grads' DataFrame
grads_cleaned = grads.copy()

# Inspect missing values before imputation
print("--- Missing values before imputation ---")
print(grads_cleaned.isnull().sum())

# Impute missing values in 'GPA' with the median GPA
median_gpa = grads_cleaned['GPA'].median()
grads_cleaned['GPA'].fillna(median_gpa, inplace=True)
print(f"\nFilled missing 'GPA' values with the median: {median_gpa:.2f}")

# Impute missing values in 'Sense of Humour (0-5)' with the median
median_humour = grads_cleaned['Sense of Humour (0-5)'].median()
grads_cleaned['Sense of Humour (0-5)'].fillna(median_humour, inplace=True)
print(f"Filled missing 'Sense of Humour (0-5)' values with the median: {median_humour:.2f}")

# Convert 'Salary' column to numeric, removing '$' and 'k' and handling missing values
# This step was done in the previous cell, but to ensure consistency if this cell is run independently,
# or if the original 'grads' was used, it's re-applied here before checking for Salary NaNs if any.
# Assuming any original 'Salary' NaN would have been converted to numeric NaN. If a salary was 'NaN',
# it would remain NaN after conversion. However, based on the `grads` DataFrame, there are no 'NaN'
# in the original 'Salary' column, only '$0k' which becomes 0.0. The `grads_outliers` copy already handled this.
# For this question, we focus on the explicit NaNs in GPA and Sense of Humour.
# If there were NaNs in Salary after conversion, median imputation would also be appropriate.

print("\n--- DataFrame after imputation ---")
print(grads_cleaned)

print("\n--- Missing values after imputation ---")
print(grads_cleaned.isnull().sum())

print("\nJustification: \n1. **Imputation over Deletion**: Given the small dataset size (9 rows), deleting rows with missing data would lead to a significant loss of valuable information (3 out of 9 rows, or 33%). Deleting columns is not justified as no column has an excessively high percentage of missing values. Therefore, imputation is preferred.\n2. **Median Imputation**: For numerical features like 'GPA' and 'Sense of Humour (0-5)', the median is chosen over the mean. The median is more robust to outliers and skewed distributions, which is particularly important in small datasets. This ensures that the imputed values do not unduly influence the distribution or statistical properties of the data.\n")


--- Missing values before imputation ---
Student Name                  0
Avg Hours Studies per Week    0
GPA                           1
University                    0
Sense of Humour (0-5)         2
Salary                        0
dtype: int64

Filled missing 'GPA' values with the median: 3.10
Filled missing 'Sense of Humour (0-5)' values with the median: 2.00

--- DataFrame after imputation ---
  Student Name  Avg Hours Studies per Week  GPA     University  \
0       George                          20  3.1            NYU   
1        Jerry                          35  3.5       Columbia   
2       Elaine                          55  4.0       Columbia   
3        Cosmo                           5  2.0   City College   
4       Newman                          25  2.8   City College   
5        Frank                          35  3.0   Festivus Uni   
6      Estelle                         100  3.2  Festivus Uni    
7          Leo                          15  2.4   Festivus Uni   
8    

/tmp/ipykernel_3493/2889403798.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  grads_cleaned['GPA'].fillna(median_gpa, inplace=True)
/tmp/ipykernel_3493/2889403798.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)

Question 3: Reload the data and fill-in the data using mean method as well as the frequent method.

In [18]:
# Reload the original grads DataFrame to start fresh
grads_reloaded = pd.read_csv("https://raw.githubusercontent.com/harmanani/AAI614/main/Week%203/grads.csv")

print("--- Original Missing values in reloaded DataFrame ---")
print(grads_reloaded.isnull().sum())

# --- Mean Imputation ---
grads_mean_imputed = grads_reloaded.copy()

# Impute missing 'GPA' with the mean GPA
mean_gpa = grads_mean_imputed['GPA'].mean()
grads_mean_imputed['GPA'] = grads_mean_imputed['GPA'].fillna(mean_gpa)
print(f"\nFilled missing 'GPA' values with the mean: {mean_gpa:.2f}")

# Impute missing 'Sense of Humour (0-5)' with the mean
mean_humour = grads_mean_imputed['Sense of Humour (0-5)'].mean()
grads_mean_imputed['Sense of Humour (0-5)'] = grads_mean_imputed['Sense of Humour (0-5)'].fillna(mean_humour)
print(f"Filled missing 'Sense of Humour (0-5)' values with the mean: {mean_humour:.2f}")

print("\n--- DataFrame after Mean Imputation ---")
print(grads_mean_imputed)
print("\n--- Missing values after Mean Imputation ---")
print(grads_mean_imputed.isnull().sum())

# --- Frequent (Mode) Imputation ---
grads_mode_imputed = grads_reloaded.copy()

# Impute missing 'GPA' with the mode GPA
# .mode() can return multiple values if there's a tie, so we take the first one.
mode_gpa = grads_mode_imputed['GPA'].mode()[0]
grads_mode_imputed['GPA'] = grads_mode_imputed['GPA'].fillna(mode_gpa)
print(f"\nFilled missing 'GPA' values with the mode: {mode_gpa:.2f}")

# Impute missing 'Sense of Humour (0-5)' with the mode
mode_humour = grads_mode_imputed['Sense of Humour (0-5)'].mode()[0]
grads_mode_imputed['Sense of Humour (0-5)'] = grads_mode_imputed['Sense of Humour (0-5)'].fillna(mode_humour)
print(f"Filled missing 'Sense of Humour (0-5)' values with the mode: {mode_humour:.2f}")

print("\n--- DataFrame after Frequent (Mode) Imputation ---")
print(grads_mode_imputed)
print("\n--- Missing values after Frequent (Mode) Imputation ---")
print(grads_mode_imputed.isnull().sum())


--- Original Missing values in reloaded DataFrame ---
Student Name                  0
Avg Hours Studies per Week    0
GPA                           1
University                    0
Sense of Humour (0-5)         2
Salary                        0
dtype: int64

Filled missing 'GPA' values with the mean: 3.11
Filled missing 'Sense of Humour (0-5)' values with the mean: 2.27

--- DataFrame after Mean Imputation ---
  Student Name  Avg Hours Studies per Week     GPA     University  \
0       George                          20  3.1125            NYU   
1        Jerry                          35  3.5000       Columbia   
2       Elaine                          55  4.0000       Columbia   
3        Cosmo                           5  2.0000   City College   
4       Newman                          25  2.8000   City College   
5        Frank                          35  3.0000   Festivus Uni   
6      Estelle                         100  3.2000  Festivus Uni    
7          Leo                   

Exercise II. Run the cell below to create a new dataframe called `df_miss`.  Its first column will contain some missing values.

In [19]:
import pandas as pd
import numpy as np
import random

nrows = 10
ncols = 5

# set a seed for random number generation
np.random.seed(314)
# create an array filled with random data
data = np.array(np.random.rand(nrows, ncols))
# put the data to a pandas dataframe
df_miss = pd.DataFrame(data)
# rename the columns
df_miss.columns = ['col_'+str(ii) for ii in range(ncols)]

# randomly set some values to missing
ix0 = np.random.randint(nrows, size=3)
ix1 = np.random.randint(nrows, size=3)

df_miss['col_0'][ix0] = np.nan
df_miss['col_1'][ix1] = np.nan

print(df_miss)

      col_0     col_1     col_2     col_3     col_4
0       NaN       NaN  0.265048  0.783205  0.918001
1  0.827355       NaN  0.260480  0.911763  0.260757
2  0.766376  0.261531  0.122291  0.386006  0.840081
3       NaN       NaN  0.633110  0.584766  0.581232
4  0.677205  0.687155  0.438927  0.320927  0.570552
5       NaN  0.861074  0.834805  0.105766  0.060408
6  0.596882  0.792395  0.226356  0.535201  0.136066
7  0.372244  0.151977  0.429822  0.792706  0.406957
8  0.177850  0.909252  0.545331  0.100497  0.718721
9  0.978429  0.309776  0.260126  0.662900  0.139720


/tmp/ipykernel_3493/310065607.py:21: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_miss['col_0'][ix0] = np.nan
/tmp/ipykernel_3493/310065607.py:22: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are settin

Impute the missing values (NaN) in `col_0` (but not `col_1`) with the median.  Store the values in the dataframe by using the parameter `inplace`.  Print the dataframe.

In [20]:
median_col_0 = df_miss['col_0'].median()
df_miss['col_0'].fillna(median_col_0, inplace=True)
print(df_miss)

      col_0     col_1     col_2     col_3     col_4
0  0.677205       NaN  0.265048  0.783205  0.918001
1  0.827355       NaN  0.260480  0.911763  0.260757
2  0.766376  0.261531  0.122291  0.386006  0.840081
3  0.677205       NaN  0.633110  0.584766  0.581232
4  0.677205  0.687155  0.438927  0.320927  0.570552
5  0.677205  0.861074  0.834805  0.105766  0.060408
6  0.596882  0.792395  0.226356  0.535201  0.136066
7  0.372244  0.151977  0.429822  0.792706  0.406957
8  0.177850  0.909252  0.545331  0.100497  0.718721
9  0.978429  0.309776  0.260126  0.662900  0.139720


/tmp/ipykernel_3493/1509232022.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_miss['col_0'].fillna(median_col_0, inplace=True)


Impute the missing values in `col_1` with value 0.  Store the values in the dataframe by using the parameter `inplace`.  Print the dataframe.

In [21]:
df_miss['col_1'].fillna(0, inplace=True)
print(df_miss)

      col_0     col_1     col_2     col_3     col_4
0  0.677205  0.000000  0.265048  0.783205  0.918001
1  0.827355  0.000000  0.260480  0.911763  0.260757
2  0.766376  0.261531  0.122291  0.386006  0.840081
3  0.677205  0.000000  0.633110  0.584766  0.581232
4  0.677205  0.687155  0.438927  0.320927  0.570552
5  0.677205  0.861074  0.834805  0.105766  0.060408
6  0.596882  0.792395  0.226356  0.535201  0.136066
7  0.372244  0.151977  0.429822  0.792706  0.406957
8  0.177850  0.909252  0.545331  0.100497  0.718721
9  0.978429  0.309776  0.260126  0.662900  0.139720


/tmp/ipykernel_3493/2213679435.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_miss['col_1'].fillna(0, inplace=True)
